# TxSON 33-Station Dynamic Data Visualization

This notebook visualizes only the final authoritative Soil + MET modeling files. Each station is loaded from the already merged per-station delivery; no legacy or intermediate pipeline output is used as a fallback.

In [ ]:
from functools import lru_cache
from pathlib import Path
import importlib
import sys

import pandas as pd
import plotly.graph_objects as go

# Change only this path when using a different authoritative delivery.
AUTHORITATIVE_ROOT = Path(
    "/Users/zuncao/Project/tx-soil-moisture/modeling_data/txson33_authoritative_2026-09-07"
).expanduser().resolve()
DATASET_VERSION = AUTHORITATIVE_ROOT.name
PER_STATION_ROOT = AUTHORITATIVE_ROOT / "per_station"
VISUALIZATION_DIR = AUTHORITATIVE_ROOT.parents[1] / "data_visualization"

if not PER_STATION_ROOT.is_dir():
    raise FileNotFoundError(f"Authoritative per-station directory not found: {PER_STATION_ROOT}")
if not (VISUALIZATION_DIR / "txson33_dynamic_visualization.py").is_file():
    raise FileNotFoundError(f"Visualization helper not found: {VISUALIZATION_DIR}")

sys.path.insert(0, str(VISUALIZATION_DIR))
import txson33_dynamic_visualization as visualization
importlib.reload(visualization)

SOIL_MOISTURE_COLS = ["SWC_5", "SWC_10", "SWC_20", "SWC_50"]
SOIL_TEMPERATURE_COLS = ["T_5", "T_10", "T_20", "T_50"]
MET_COLS = ["Ppt", "Tair", "RH", "Srad", "Wind speed", "Wind direction"]
ROW_PROVENANCE_COLS = ["Any_Soil_Imputed", "Any_MET_Imputed"]


## Authoritative loader

The loader requires all 33 final per-station files and parses `Timestamp` as the unique, monotonic datetime index. Columns that are structurally unavailable remain NaN in the delivery and are reported as unavailable by the plot. The two provenance fields are row-level flags, so they are retained in memory but are not used to label an individual parameter as imputed.

In [ ]:
def discover_authoritative_stations() -> list[str]:
    stations = sorted(
        path.name.removeprefix("Station").removesuffix("_modeling.csv")
        for path in PER_STATION_ROOT.glob("Station*_modeling.csv")
    )
    if len(stations) != 33 or len(set(stations)) != 33:
        raise FileNotFoundError(
            f"Expected 33 authoritative station files in {PER_STATION_ROOT}; found {len(stations)}"
        )
    return stations


@lru_cache(maxsize=33)
def _load_authoritative_station(station_id: str) -> pd.DataFrame:
    if station_id not in discover_authoritative_stations():
        raise ValueError(f"Unknown authoritative station: {station_id}")
    path = PER_STATION_ROOT / f"Station{station_id}_modeling.csv"
    frame = pd.read_csv(path, parse_dates=["Timestamp"], low_memory=False)
    required = {"Station", "Timestamp"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{path} is missing required columns: {sorted(required - set(frame.columns))}")
    station_values = set(frame["Station"].dropna().astype(str))
    if station_values != {station_id}:
        raise ValueError(f"{path} contains unexpected station values: {sorted(station_values)}")
    if frame["Timestamp"].isna().any():
        raise ValueError(f"{path} contains invalid timestamps")
    frame = frame.set_index("Timestamp")
    if frame.index.has_duplicates:
        raise ValueError(f"{path} contains duplicate timestamps")
    if not frame.index.is_monotonic_increasing:
        raise ValueError(f"{path} timestamps are not monotonic")
    return frame


def load_authoritative_station(station_id: str) -> pd.DataFrame:
    return _load_authoritative_station(station_id).copy()


def _empty_figure(title: str, message: str) -> go.Figure:
    figure = go.Figure()
    figure.update_layout(title=title, template="plotly_white", autosize=True, height=560)
    figure.add_annotation(
        text=message, x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False
    )
    return figure


def build_authoritative_figure(
    station_id: str,
    columns: list[str],
    year: int,
    month: int,
    time_resolution: str,
    label: str,
    yaxis_title: str,
) -> go.Figure:
    complete = load_authoritative_station(station_id)
    structurally_available = [
        column for column in columns
        if column in complete.columns and complete[column].notna().any()
    ]
    structurally_unavailable = [column for column in columns if column not in structurally_available]
    period = visualization.period_label(year, month, time_resolution)
    title = f"{label} for Station {station_id} {period}<br><sup>{DATASET_VERSION}</sup>"
    if not structurally_available:
        return _empty_figure(
            title,
            f"{', '.join(columns)} is unavailable for Station {station_id} in the authoritative dataset",
        )

    selected = visualization.filter_time(complete, year, month, time_resolution)
    period_available = [
        column for column in structurally_available
        if column in selected.columns and selected[column].notna().any()
    ]
    if selected.empty or not period_available:
        return _empty_figure(title, "No available values for the selected time period")

    figure = visualization.make_line_figure(
        selected, station_id, period_available, title, yaxis_title, show_missing=True
    )
    figure.update_layout(autosize=True, height=560)
    if structurally_unavailable:
        figure.add_annotation(
            text=f"Unavailable for this station: {', '.join(structurally_unavailable)}",
            x=1, y=1.08, xref="paper", yref="paper", xanchor="right",
            showarrow=False, font={"size": 11, "color": "#666666"},
        )
    return figure


def plot_authoritative_soil_moisture(station_id, year, month, time_resolution):
    figure = build_authoritative_figure(
        station_id, SOIL_MOISTURE_COLS, year, month, time_resolution,
        "Soil Moisture", "Soil Moisture (m3/m3)",
    )
    figure.show(config={"responsive": True})
    return figure


def plot_authoritative_soil_temperature(station_id, year, month, time_resolution):
    figure = build_authoritative_figure(
        station_id, SOIL_TEMPERATURE_COLS, year, month, time_resolution,
        "Soil Temperature", "Temperature (C)",
    )
    figure.show(config={"responsive": True})
    return figure


def plot_authoritative_met(station_id, variable, year, month, time_resolution):
    figure = build_authoritative_figure(
        station_id, [variable], year, month, time_resolution, variable, variable
    )
    figure.show(config={"responsive": True})
    return figure


# Reuse the existing dashboard controls and plot styling with the strict final-data loader.
visualization.discover_stations = discover_authoritative_stations
visualization.load_station_data = load_authoritative_station
visualization.plot_soil_moisture = plot_authoritative_soil_moisture
visualization.plot_soil_temperature = plot_authoritative_soil_temperature
visualization.plot_met_variable = plot_authoritative_met
display_dashboard = visualization.display_dashboard

print(f"Dataset: {DATASET_VERSION}")
print(f"Per-station source: {PER_STATION_ROOT}")
print(f"Stations available: {len(discover_authoritative_stations())}")


## Interactive plot

Use Plot Type, Station, Year, Month, MET Variable, and Time Type. Missing 50-cm sensors and non-Ppt MET variables without independent coverage are labeled unavailable instead of being interpreted as failed imputation. Every title includes the authoritative dataset version.

In [ ]:
dashboard_ui, dashboard_output = display_dashboard()
dashboard_ui.layout.width = "100%"
dashboard_output.layout.width = "100%"
